# Práctica de detección de objetos

### De: Lucas Amado Oseira y María Lestón Geada


##### **Dataset:** Lettuce Pallets
##### **Moldelos:** Cascade R-CNN (Detectron2) y SSD300-VGG16 (Torchvision)


In [16]:
from google.colab import drive
import tarfile, random, torch
import numpy as np
from pathlib import Path

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

In [8]:
SEED = 2710

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [10]:
drive.mount('/content/drive')

DATA_ARCHIVE =  "/content/drive/MyDrive/dataset.tar.gz"
DATA_ROOT = Path("dataset")

DATA_ROOT.mkdir(exist_ok=True)

if not any(DATA_ROOT.iterdir()):
    with tarfile.open(DATA_ARCHIVE, "r:gz") as tar:
        tar.extractall(DATA_ROOT)

print(f"Dataset extracted to: {DATA_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_17074/3343367773.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(DATA_ROOT)


Dataset extracted to: dataset


In [15]:
json_files = list(DATA_ROOT.glob("**/*.json"))
print(f"JSON encontrados: {len(json_files)}")
for path in json_files:
    print(f" - {path.relative_to(DATA_ROOT)}")

def split_token(path):
    s = str(path).lower()
    if "train" in s: return "train"
    elif "val" in s or "valid" in s: return "val"
    elif "test" in s: return "test"
    return None

ANN_FILES = {}

for split in ["train", "val", "test"]:
    candidates = [ p for p in json_files if split_token(p) == split]
    if candidates:
        ANN_FILES[split] = candidates[0]
        print(f"{split}: {ANN_FILES[split].relative_to(DATA_ROOT)}")

IMAGE_DIRS = {}
for split in ["train", "val", "test"]:
    candidates = [ p for p in DATA_ROOT.glob("**/*") if p.is_dir() and split_token(p) == split]
    if candidates:
        IMAGE_DIRS[split] = candidates[0]
        print(f"{split}: {IMAGE_DIRS[split].relative_to(DATA_ROOT)}")

print("ANN_FILES:", ANN_FILES)
print("IMAGE_DIRS:", IMAGE_DIRS)


JSON encontrados: 3
 - home/zuppif/Documents/Work/RoboFlow/ODinW-RF100-challenge/rf100/lettuce-pallets/test/_annotations.coco.json
 - home/zuppif/Documents/Work/RoboFlow/ODinW-RF100-challenge/rf100/lettuce-pallets/valid/_annotations.coco.json
 - home/zuppif/Documents/Work/RoboFlow/ODinW-RF100-challenge/rf100/lettuce-pallets/train/_annotations.coco.json
train: home/zuppif/Documents/Work/RoboFlow/ODinW-RF100-challenge/rf100/lettuce-pallets/train/_annotations.coco.json
val: home/zuppif/Documents/Work/RoboFlow/ODinW-RF100-challenge/rf100/lettuce-pallets/valid/_annotations.coco.json
test: home/zuppif/Documents/Work/RoboFlow/ODinW-RF100-challenge/rf100/lettuce-pallets/test/_annotations.coco.json
train: home/zuppif/Documents/Work/RoboFlow/ODinW-RF100-challenge/rf100/lettuce-pallets/train
val: home/zuppif/Documents/Work/RoboFlow/ODinW-RF100-challenge/rf100/lettuce-pallets/valid
test: home/zuppif/Documents/Work/RoboFlow/ODinW-RF100-challenge/rf100/lettuce-pallets/test
ANN_FILES: {'train': Posix

In [17]:
coco_train = COCO(str(ANN_FILES["train"]))
cats = sorted(coco_train.loadCats(coco_train.getCatIds()), key=lambda x: x["id"])
print(f"Categorías COCO: ", [(c["id"], c["name"]) for c in cats])
print(f"Total imágenes de entrenamiento: {len(coco_train.getImgIds())}")
print(f"Número de anotaciones train: {len(coco_train.getAnnIds())}")

loading annotations into memory...
Done (t=0.04s)
creating index...
index created!
Categorías COCO:  [(0, 'lettuce'), (1, 'Ready'), (2, 'empty_pod'), (3, 'germination'), (4, 'pod'), (5, 'young')]
Total imágenes de entrenamiento: 1060
Número de anotaciones train: 13812
